<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/lab-00-run-the-harness-both-ways.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 0 (ungraded) — Run the harness, both ways
**Course 3: AI Agents and Agentic AI with Python — Chapter 0: Onboarding to the agents group**

Goal: run a minimal agent harness against a real API, then against a fully offline
deterministic mock, and confirm both produce a sensible, auditable trace before Chapter 1.

## 1. Two real tools + their deterministic mocks
Every real-tool lab in this course ships a deterministic mock so no lab ever requires a key
or connectivity. Here's the pattern you'll reuse throughout the course.

In [ ]:
import urllib.request
import json

def get_weather(city):
    try:
        url = f'https://wttr.in/{city}?format=j1'
        with urllib.request.urlopen(url, timeout=5) as resp:
            data = json.load(resp)
        temp_c = data['current_condition'][0]['temp_C']
        return f'{city}: {temp_c}°C, {data["current_condition"][0]["weatherDesc"][0]["value"]}'
    except Exception as e:
        print(f'  (real weather API unavailable: {e} — using the deterministic mock)')
        return f'{city}: 18°C, Partly cloudy (mock data)'

def calculator(expression):
    """A genuinely real, always-available tool — no network dependency at all."""
    allowed = set('0123456789+-*/(). ')
    if not set(expression) <= allowed:
        return 'error: expression contains disallowed characters'
    return str(eval(expression))

print(get_weather('London'))
print('12 * (4 + 3) =', calculator('12 * (4 + 3)'))

## 2. A minimal harness: one tool call, traced

In [ ]:
import time

TOOLS = {'get_weather': get_weather, 'calculator': calculator}

def run_traced_tool_call(tool_name, arg):
    t0 = time.perf_counter()
    result = TOOLS[tool_name](arg)
    trace = {
        'tool': tool_name, 'arg': arg, 'result': result,
        'latency_ms': round((time.perf_counter() - t0) * 1000, 1),
    }
    return trace

trace1 = run_traced_tool_call('get_weather', 'Nairobi')
trace2 = run_traced_tool_call('calculator', '(100 - 15) / 5')
for t in (trace1, trace2):
    print(t)

## 3. Hosted-LLM path (optional) vs. fully offline

In [ ]:
import os
try:
    from google.colab import userdata
    API_KEY = userdata.get('LLM_API_KEY'); BASE_URL = userdata.get('LLM_BASE_URL')
except Exception:
    API_KEY = os.environ.get('LLM_API_KEY'); BASE_URL = os.environ.get('LLM_BASE_URL')
hosted_available = bool(API_KEY and BASE_URL)
print('Hosted LLM available for this course:', hosted_available)
print('Every agent lab in this course works with this set to False — confirm that stays true')
print('as you go by occasionally testing with your key unset.')

---
*Beacon AI · AIBits Academy — Chapter 0: Onboarding to the agents group*